# Sorting & Binary Search

*All Sorts · Stability · Timsort · bisect · Real-World*


---
## Sorting & Binary Search


# Sorting Searching

*Run each cell with **Shift+Enter***

01 — DSA Masterclass: Sorting & Searching
=========================================

Runnable companion to PDF Chapter "1+ — DSA Masterclass"
(sorting & binary-search sections).

  * Bubble, insertion, selection sort  -> O(n^2) family
  * Merge sort  -> stable, guaranteed O(n log n)
  * Quick sort  -> avg O(n log n), worst O(n^2)
  * Binary search  -> O(log n) on sorted data (+ bisect variants)

Each sort is verified against Python's built-in `sorted`.

Run:  python sorting_searching.py


---
## 🧠 Mental Model: Sorting & Binary Search

> **Sorting is the key that unlocks O(log n) search, O(n) deduplication, and O(n) median.**  
> Unsorted data → O(n) lookup. Sorted data → O(log n) lookup. Paying O(n log n) to sort once is almost always worth it.

### WHY — Why do multiple sorts exist?
No single sort dominates all scenarios. Mergesort is stable and guaranteed; quicksort is faster in practice (cache-friendly, in-place); Python's Timsort exploits existing runs in nearly-sorted data.

### WHAT & HOW — Algorithm comparison

| Algorithm | Time (avg) | Time (worst) | Space | Stable? | Notes |
|-----------|-----------|-------------|-------|---------|-------|
| Bubble sort | O(n²) | O(n²) | O(1) | ✓ | Only for teaching |
| Insertion sort | O(n²) | O(n²) | O(1) | ✓ | Fast for small/nearly-sorted |
| Selection sort | O(n²) | O(n²) | O(1) | ✗ | Never use |
| **Merge sort** | O(n log n) | O(n log n) | O(n) | ✓ | Guaranteed; good for linked lists |
| **Quick sort** | O(n log n) | O(n²) | O(log n) | ✗ | Fastest in practice; pivot choice matters |
| **Timsort** (Python) | O(n log n) | O(n log n) | O(n) | ✓ | Python's `sorted()` / `.sort()`; exploits runs |
| Heap sort | O(n log n) | O(n log n) | O(1) | ✗ | In-place; poor cache locality |
| Counting/Radix sort | O(n+k) | O(n+k) | O(k) | ✓ | When keys are bounded integers |

**Stability** = equal elements maintain original relative order → crucial when sorting by secondary then primary key.

### Binary Search — the invariant

```python
# Classic binary search template
lo, hi = 0, len(arr) - 1
while lo <= hi:
    mid = (lo + hi) // 2
    if arr[mid] == target:
        return mid
    elif arr[mid] < target:
        lo = mid + 1   # target is in right half
    else:
        hi = mid - 1   # target is in left half
```

**Python's `bisect` module:**
- `bisect_left(a, x)` → index of leftmost position to insert x (first x if present)
- `bisect_right(a, x)` → index of rightmost position to insert x (after last x if present)

### WHEN — Decision guide

| Scenario | Best choice |
|----------|------------|
| General-purpose sort | `sorted()` / `.sort()` (Timsort) |
| Need guaranteed O(n log n) worst-case | Merge sort or Timsort |
| Sort with minimal memory | Heap sort |
| Keys are small non-negative integers | Counting / Radix sort |
| Find element in sorted array | Binary search / `bisect` |
| Find insertion point | `bisect_left` / `bisect_right` |

**Gotchas:**
- `list.sort()` modifies in place and returns `None` — `result = lst.sort()` is a common bug.
- Quick sort worst case O(n²) happens on sorted/reverse-sorted input without random pivot.
- Binary search requires **sorted** input — always verify this precondition.
- Off-by-one: `lo <= hi` (not `<`) when searching for exact match.


In [ ]:
from __future__ import annotations

import bisect
import random

===========================================================================
O(n^2) FAMILY
===========================================================================

In [ ]:
def bubble_sort(a: list[int]) -> list[int]:
    a = a[:]
    for i in range(len(a)):
        swapped = False
        for j in range(len(a) - 1 - i):
            if a[j] > a[j + 1]:
                a[j], a[j + 1] = a[j + 1], a[j]
                swapped = True
        if not swapped:  # already sorted -> best case O(n)
            break
    return a


def insertion_sort(a: list[int]) -> list[int]:
    a = a[:]
    for i in range(1, len(a)):
        key, j = a[i], i - 1
        while j >= 0 and a[j] > key:  # shift the sorted prefix right
            a[j + 1] = a[j]
            j -= 1
        a[j + 1] = key
    return a


def selection_sort(a: list[int]) -> list[int]:
    a = a[:]
    for i in range(len(a)):
        smallest = i
        for j in range(i + 1, len(a)):
            if a[j] < a[smallest]:
                smallest = j
        a[i], a[smallest] = a[smallest], a[i]  # one swap per pass -> O(n) swaps
    return a

===========================================================================
O(n log n) DIVIDE & CONQUER
===========================================================================

In [ ]:
def merge_sort(a: list[int]) -> list[int]:
    """Stable, guaranteed O(n log n), O(n) extra space."""
    if len(a) <= 1:
        return a
    mid = len(a) // 2
    left, right = merge_sort(a[:mid]), merge_sort(a[mid:])
    merged: list[int] = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:  # <= keeps it STABLE
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1
    merged.extend(left[i:])
    merged.extend(right[j:])
    return merged


def quick_sort(a: list[int]) -> list[int]:
    """Average O(n log n); randomized pivot avoids the O(n^2) worst case."""
    if len(a) <= 1:
        return a
    pivot = a[random.randint(0, len(a) - 1)]
    less = [x for x in a if x < pivot]
    equal = [x for x in a if x == pivot]
    greater = [x for x in a if x > pivot]
    return quick_sort(less) + equal + quick_sort(greater)

===========================================================================
BINARY SEARCH — requires SORTED input, O(log n)
===========================================================================

In [ ]:
def binary_search(a: list[int], target: int) -> int:
    lo, hi = 0, len(a) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if a[mid] == target:
            return mid
        if a[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1


def first_ge(a: list[int], target: int) -> int:
    """Leftmost index with a[i] >= target — 'binary search on the boundary'."""
    return bisect.bisect_left(a, target)


def main() -> None:
    print("=" * 68)
    print("DSA MASTERCLASS — sorting_searching.py")
    print("=" * 68)

    sorts = [bubble_sort, insertion_sort, selection_sort, merge_sort, quick_sort]
    for _ in range(200):  # property test against the built-in
        data = [random.randint(-50, 50) for _ in range(random.randint(0, 30))]
        expected = sorted(data)
        for fn in sorts:
            assert fn(data) == expected, fn.__name__
    print("all 5 sorts match sorted() over 200 random cases ✔")

    arr = list(range(0, 100, 2))  # 0,2,4,...,98
    assert binary_search(arr, 42) == 21
    assert binary_search(arr, 43) == -1
    assert first_ge(arr, 43) == 22  # first index whose value >= 43 is 44 @ idx 22
    print("binary search: found 42 @ idx 21; 43 absent; first>=43 @ idx 22")

    print("-" * 68)
    print("All sorting/searching demos passed ✔")

## ═══  EXHAUSTIVE NOTEBOOK — sorting & searching gotchas  ═══════════════════

In [ ]:
import sys, time

def sep(t): print(f"\n{'═'*64}\n  {t}\n{'═'*64}")

## NOTEBOOK §S1 — SORTING ALGORITHM GUIDE & GOTCHAS

Algorithm Decision Table

Bubble sort    O(n²)      Never use in production.  Pedagogical only.
Insertion sort O(n²) avg  GOOD for small n (< ~50) or nearly-sorted data.
                          Python's Timsort uses it for small runs.
Selection sort O(n²)      O(n) swaps — useful when writes are expensive.
                          NOT stable.
Merge sort     O(n log n) Stable, guaranteed. Best for linked lists.
                          O(n) extra space. Used in Python's Timsort.
Quick sort     O(n log n) avg, O(n²) worst.  Best cache performance.
                          Randomised pivot avoids worst case in practice.
Python sorted  O(n log n) Timsort (merge + insertion). STABLE. USE THIS.

> ⚠️ **GOTCHA 1: "Stable" means equal elements keep their ORIGINAL ORDER.**
  Critical when sorting objects by one key when another key already orders them.
> ⚠️ **GOTCHA 2: Python's built-in sort is ALWAYS your first choice.**
  Custom sorts are only justified for very specific constraints (linked list,
  external/streaming, special comparators, constant extra space).
> ⚠️ **GOTCHA 3: Quick sort's worst case is O(n²) on SORTED OR REVERSE-SORTED input**
  with a fixed pivot. Always use a RANDOMISED pivot in production.
> ⚠️ **GOTCHA 4: Merge sort uses O(n) extra space — relevant for large datasets.**

In [ ]:
def notebook_sorting_gotchas() -> None:

## §S1 · Sorting Gotchas

In [ ]:
# ── §S1.1  Stability matters for multi-key sorting ─────────────────────
    people = [
        ("Alice", 30), ("Bob", 25), ("Carol", 30), ("Dave", 25)
    ]
    # Sort by age first, then by name — classic multi-key stable sort
    by_age = sorted(people, key=lambda p: p[1])   # stable: Dave before Bob within age=25
    print("Stable sort by age:", by_age)
    # Bob and Dave both have age=25; they appear in original order (Alice,Bob,Carol,Dave)
    assert by_age.index(("Bob",25)) < by_age.index(("Dave",25))
    print("  Original insertion order preserved within equal keys ✓")

    # Sorting tuples: Python compares element-by-element (lexicographic)
    tasks = [("high", 3), ("low", 1), ("med", 2), ("high", 1)]
    print("\nTuple sort (first element, then second):", sorted(tasks))
    # [("high",1), ("high",3), ("low",1), ("med",2)]

    # ── §S1.2  GOTCHA: quick sort worst case on sorted input ──────────────
    import random, time

    def quick_sort_fixed_pivot(a):
        """DANGEROUS: always picks last element as pivot."""
        if len(a) <= 1: return a
        pivot = a[-1]   # fixed pivot = disaster on sorted input
        less    = [x for x in a[:-1] if x <= pivot]
        greater = [x for x in a[:-1] if x  > pivot]
        return quick_sort_fixed_pivot(less) + [pivot] + quick_sort_fixed_pivot(greater)

    n = 500   # small n to avoid RecursionError with the bad version
    sorted_data = list(range(n))

    t0 = time.perf_counter()
    try:
        quick_sort_fixed_pivot(sorted_data)
        t_bad = time.perf_counter() - t0
        print(f"\nFixed-pivot quicksort on sorted({n}): {t_bad*1000:.1f}ms (O(n²))")
    except RecursionError:
        print(f"\nFixed-pivot quicksort on sorted({n}): RecursionError! (stack overflow)")

    t0 = time.perf_counter()
    quick_sort(sorted_data)   # randomised pivot version
    t_good = time.perf_counter() - t0
    print(f"Random-pivot quicksort on sorted({n}): {t_good*1000:.1f}ms (O(n log n))")

    # ── §S1.3  Python's sort vs custom ────────────────────────────────────
    data = [random.randint(0, 1000) for _ in range(10_000)]

    t0 = time.perf_counter()
    sorted(data)
    t_builtin = time.perf_counter() - t0

    t0 = time.perf_counter()
    merge_sort(data)
    t_custom = time.perf_counter() - t0

    print(f"\nSort 10,000 random ints:")
    print(f"  Python sorted():  {t_builtin*1000:.2f}ms")
    print(f"  Custom merge_sort:{t_custom*1000:.2f}ms")
    print(f"  Always use Python's built-in — it's Timsort in C")

    # ── §S1.4  Key function vs cmp_to_key ─────────────────────────────────
    from functools import cmp_to_key

    # Sorting by a computed property — key is more efficient (called once per element)
    words = ["banana", "Apple", "cherry", "date", "Fig"]
    by_len_then_alpha = sorted(words, key=lambda w: (len(w), w.lower()))
    print(f"\nSort by length then alpha: {by_len_then_alpha}")

    # cmp function needed for complex comparisons (e.g., largest number from digits)
    def largest_number_cmp(a, b):
        """Compare: which ordering makes the larger concatenated number?"""
        if a + b > b + a: return -1   # a should come first
        if a + b < b + a: return 1
        return 0

    nums = ["3", "30", "34", "5", "9"]
    result = "".join(sorted(nums, key=cmp_to_key(largest_number_cmp)))
    print(f"Largest number from {nums}: {result}")   # 9534330

## NOTEBOOK §S2 — BINARY SEARCH GOTCHAS

Mental model

Binary search halves the search space each step: O(log n).
REQUIRES: the input must be SORTED and support O(1) indexing.

> ⚠️ **GOTCHA 1: Integer overflow in `mid = (lo + hi) // 2`.**
  (Not a Python problem — Python ints are arbitrary precision.
   IS a problem in Java/C++: use `lo + (hi - lo) // 2` for safety.)
> ⚠️ **GOTCHA 2: Loop condition `lo <= hi` vs `lo < hi` — different semantics!**
  lo <= hi: searches inclusive range; terminates when lo > hi.
  lo < hi:  searches exclusive range; terminates when lo == hi.
> ⚠️ **GOTCHA 3: Off-by-one in the return: target at `hi` when `lo <= hi` terminates.**
> ⚠️ **GOTCHA 4: bisect_left vs bisect_right — they differ on DUPLICATE elements.**
  bisect_left:  returns leftmost position where target CAN be inserted.
  bisect_right: returns rightmost position (just after existing occurrences).

In [ ]:
def notebook_binary_search_gotchas() -> None:

## §S2 · Binary Search Gotchas

In [ ]:
# ── §S2.1  Standard binary search ─────────────────────────────────────
    arr = list(range(0, 20, 2))   # [0,2,4,...,18]
    assert binary_search(arr, 8) == 4
    assert binary_search(arr, 7) == -1
    print("binary_search works ✓")

    # ── §S2.2  GOTCHA: bisect_left vs bisect_right on duplicates ──────────
    dup = [1, 2, 2, 2, 3, 4]
    left  = bisect.bisect_left(dup,  2)   # leftmost position of 2
    right = bisect.bisect_right(dup, 2)   # just past the last 2

    print(f"\nbisect on {dup} for target 2:")
    print(f"  bisect_left  = {left}  (index of first 2)")
    print(f"  bisect_right = {right}  (index after last 2)")
    print(f"  All 2s at indices: {list(range(left, right))}")

    # Count occurrences in O(log n)
    count = right - left
    print(f"  Occurrences of 2: {count}")

    # ── §S2.3  First/last occurrence ──────────────────────────────────────
    def first_occurrence(arr, target):
        idx = bisect.bisect_left(arr, target)
        return idx if idx < len(arr) and arr[idx] == target else -1

    def last_occurrence(arr, target):
        idx = bisect.bisect_right(arr, target) - 1
        return idx if idx >= 0 and arr[idx] == target else -1

    data = [1, 2, 2, 2, 3, 4]
    print(f"\nFirst occurrence of 2 in {data}: {first_occurrence(data, 2)}")
    print(f"Last  occurrence of 2 in {data}: {last_occurrence(data, 2)}")

    # ── §S2.4  Binary search on the ANSWER (search space != array index) ──
    #
    # Pattern: "find the minimum X such that condition(X) is True"
    # The search space is a RANGE OF VALUES, not array indices.

    def min_days_to_make_bouquets(bloomDay, m, k):
        """
        Given bloomDay[i] = day flower i blooms, find minimum days needed
        to make m bouquets each requiring k consecutive bloomed flowers.
        Binary search on the answer: day in range [min(bloomDay), max(bloomDay)].
        """
        if m * k > len(bloomDay): return -1

        def can_make(day):
            bouquets = consecutive = 0
            for d in bloomDay:
                if d <= day:
                    consecutive += 1
                    if consecutive == k:
                        bouquets += 1; consecutive = 0
                else:
                    consecutive = 0
            return bouquets >= m

        lo, hi = min(bloomDay), max(bloomDay)
        while lo < hi:
            mid = (lo + hi) // 2
            if can_make(mid): hi = mid    # mid works, try earlier
            else:             lo = mid + 1  # too early, need more days
        return lo

    result = min_days_to_make_bouquets([1,10,3,10,2], m=3, k=1)
    print(f"\nBinary search on answer: min days for bouquets = {result}")   # 3


def run_sorting_notebook() -> None:
    notebook_sorting_gotchas()
    notebook_binary_search_gotchas()
    print("\n" + "═"*64)
    print("  SORTING & SEARCHING NOTEBOOK COMPLETE")
    print("═"*64)


if __name__ == "__main__":
    if hasattr(sys.stdout, "reconfigure"):
        sys.stdout.reconfigure(encoding="utf-8")
    main()
    run_sorting_notebook()

---
## Exhaustive Gotchas & Real-World Scenarios


ShopFlow's search bar made a database query on EVERY keystroke.
At 50k products × 1000 concurrent users: 50M DB reads per minute.
The fix: keep a sorted list of product names in memory, binary search
on keystrokes. O(log n) instead of O(n) per keystroke.
But binary search requires SORTED input — this rule is often forgotten.

In [ ]:
def scenario_binary_search() -> None:

In [ ]:
import bisect

    h("WITHOUT binary search — O(n) per keystroke")
    products = [f"product_{i:06d}" for i in range(50_000)]
    query    = "product_04"

    t0 = time.perf_counter()
    for _ in range(100):
        matches_linear = [p for p in products if p.startswith(query)]
    t_linear = time.perf_counter() - t0

    h("WITH binary search — O(log n) to find the insertion point")
    sorted_products = sorted(products)     # sort once at startup

    def prefix_search(sorted_list: list[str], prefix: str) -> list[str]:
        """Find all strings with the given prefix using binary search."""
        lo = bisect.bisect_left(sorted_list, prefix)
        # Upper bound: prefix with last char incremented (next lexicographic string)
        hi = bisect.bisect_left(sorted_list, prefix[:-1] + chr(ord(prefix[-1]) + 1))
        return sorted_list[lo:hi]

    t0 = time.perf_counter()
    for _ in range(100):
        matches_binary = prefix_search(sorted_products, query)
    t_binary = time.perf_counter() - t0

    assert set(matches_linear) == set(matches_binary)
    print(f"Linear scan (O(n)):    {t_linear*1000:.1f}ms for 100 keystrokes")
    print(f"Binary search (O(logn):{t_binary*1000:.1f}ms for 100 keystrokes")
    print(f"Speedup: ~{t_linear/max(t_binary,0.0001):.0f}×, matches found: {len(matches_binary)}")

    h("GOTCHA: binary search requires SORTED input — common mistake")
    unsorted = [5, 3, 1, 4, 2]
    # bisect assumes sorted — gives WRONG results on unsorted input!
    wrong_idx = bisect.bisect_left(unsorted, 3)
    correct_idx = bisect.bisect_left(sorted(unsorted), 3)
    print(f"\nUnsorted bisect(3) → {wrong_idx} (WRONG, should be index of 3 in sorted = {correct_idx})")
    print("RULE: always verify your list is sorted before using bisect")

    h("bisect_left vs bisect_right — counts and ranges")
    sorted_scores = [60, 70, 70, 70, 80, 90, 100]
    left  = bisect.bisect_left(sorted_scores, 70)
    right = bisect.bisect_right(sorted_scores, 70)
    print(f"\nbisect_left(70)  = {left}  (index of FIRST 70)")
    print(f"bisect_right(70) = {right}  (index AFTER LAST 70)")
    print(f"Count of 70s = right - left = {right - left}")
    print(f"Rank of score 75: {bisect.bisect_right(sorted_scores, 75)} out of {len(sorted_scores)}")

    h("WHERE binary search is used in frameworks")
    print("""
  Python bisect module: bisect_left/right for sorted list operations
  sortedcontainers.SortedList: O(log n) add/remove, O(1) index
  Database B-tree indexes: every WHERE/ORDER BY on an indexed column
    uses binary search internally (O(log n) instead of O(n) table scan)
  Git bisect: binary search through commit history to find a bug
  NumPy searchsorted: vectorized binary search on arrays
""")